# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [1]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt

import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [2]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [3]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https://edwarddonner.com/2026/01/04/ai-builder-with-n8n-create-agents-and-voice-agents/',
 'https:/

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [4]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [5]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [7]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/
https://edwarddonner.com/2026/02/17

In [14]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [15]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'homepage', 'url': 'https://edwarddonner.com/'},
  {'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'blog', 'url': 'https://edwarddonner.com/posts/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/avatar/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/curriculum/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/proficient/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/connect-four/'},
  {'type': 'portfolio page', 'url': 'https://edwarddonner.com/outsmart/'},
  {'type': 'company page',
   'url': 'https://nebula.io/?utm_source=ed&utm_medium=referral'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'},
  {'type': 'blog',
   'url': 'https://edwarddonner.com/2026/02

In [16]:
select_relevant_links("https://huggingface.co")

{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub page', 'url': 'https://github.com/huggingface'},
  {'type': 'LinkedIn page',
   'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Twitter page', 'url': 'https://twitter.com/huggingface'},
  {'type': 'Discord community', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Discuss forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'Status page', 'url': 'https://status.huggingface.co/'}]}

In [17]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [18]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 5 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'home page', 'url': 'https://edwarddonner.com/'},
  {'type': 'linkedin', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'twitter', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'facebook', 'url': 'https://www.facebook.com/edward.donner.52'}]}

In [19]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


{'links': [{'type': 'about page', 'url': 'https://huggingface.co/brand'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'discussion forum', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'join Discord', 'url': 'https://huggingface.co/join/discord'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'endpoints', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'status page', 'url': 'https://status.huggingface.co/'}]}

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [23]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [12]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Community
Docs
Enterprise
Pricing
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
zai-org/GLM-OCR
Updated
4 days ago
•
204k
•
743
moonshotai/Kimi-K2.5
Updated
2 days ago
•
335k
•
1.8k
Qwen/Qwen3-Coder-Next
Updated
4 days ago
•
53.5k
•
544
stepfun-ai/Step-3.5-Flash
Updated
about 3 hours ago
•
12k
•
493
circlestone-labs/Anima
Updated
6 days ago
•
60.6k
•
484
Browse 2M+ models
Spaces
Running
on
Zero
Featured
1.28k
Qwen3-TTS Demo
🎙
1.28k
Transform text into natural-sounding speech with custom voices
Running
on
A100
162
ACE-Step v1.5
🎵
162
Music Generation Foundation Model v1.5
Running
463
Demo Playground
⚡
463
Free platform to access multiple AI models
Running
on
Zero
MCP
2.04k
Z Image Turbo
🖼
2.04k
Generate stunn

In [20]:
# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

brochure_system_prompt = """
You are a snarky assistant that analyzes the contents of several relevant pages from a company website
and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""


In [21]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [24]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


'\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\nQwen/Qwen3.8-27B\nUpdated\n6 days ago\n•\n1.37M\n•\n11.6k\nunsloth/Qwen3.8-27B-GGUF\nUpdated\nabout 17 hours ago\n•\

In [25]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [26]:
create_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 9 relevant links


# 🤗 Hugging Face: The AI Community Building the Future (And Maybe Your Next Best Friend)

---

## Who Are We?

Welcome to **Hugging Face**, the buzzing hive where the machine learning community swarms to collaborate, create, and caffeinate (well, we imagine). We’re not your grandma’s AI company — we’re your open, friendly, and borderline snarky platform where **millions of models, datasets, and apps** are born, bred, and battle-tested.

Whether you’re a data scientist, a curious coder, or just someone who likes playing with AI that *actually* works, Hugging Face is your playground. Explore **2 million+ models**, **500k+ datasets**, and over **1 million AI applications** — all at your fingertips and zero gatekeeping.

---

## What Do We Do? (Besides Make AI Awesome)

- **Models, Models, and More Models:** From chatbots smarter than your cat (probably) to AI that can generate music, videos, or even edit images — we host it all. Trending right now? The Qwen series (yep, it's as classy as it sounds).
  
- **Spaces:** Your canvas for AI apps and demos. Want to create music just by typing? Check out MiniMax Music 3 Studio. Fancy generating videos from text prompts? We’ve got that too. Spoiler alert: it’s cooler than your magic wand.

- **Datasets That Don’t Suck:** Over half a million carefully curated datasets to feed your AI’s insatiable appetite.

- **Enterprise & PRO:** For the serious folks, we offer professional support, inference endpoints, storage buckets, and more — so your AI solutions can run like a dream (or at least like a really smooth Tesla).

---

## Culture: Where Nerds and Nice People Unite 🤝

At Hugging Face, we believe in **community over competition**. We’re enthusiasts, collaborators, and part-time AI whisperers who:

- Celebrate open source like it’s a national holiday.
- Encourage curiosity, creativity, and a little bit of chaos.
- Keep our Discord and Forums buzzing with developers, beginners, and AI geeks swapping ideas (and memes).
- Believe every good AI deserves a great *hug*—because what else would a company named Hugging Face do?

If you want your brain to be challenged, your passion ignited, and your memes appreciated, you’ll fit right in.

---

## Customers? We’ve Got ‘Em

Our platform is the go-to for:

- **Data scientists** who want to skip the headaches and get straight to experimenting.
- **Developers and enterprises** seeking scalable AI solutions without the vendor lock-in nightmare.
- **Educators and students** learning and sharing the future of machine learning.
- And anyone who just loves a good AI-powered surprise.

---

## Careers: Join the Hug Squad!

Think you can hang with the wizards of machine learning? Here’s what you’ll get:

- To work alongside some of the brightest, quirkiest, and most passionate minds in AI.
- The thrill of open-source innovation and direct impact on millions of users worldwide.
- A culture that values your individuality — and your terrible AI jokes.
- Competitive perks, growth opportunities, and endless coffee (actual coffee included).

You don’t need a cape, but passion for AI is mandatory.

---

## Why Hugging Face? Because We’re More Than AI — We’re a Movement

- Built on **collaboration**, powered by **community**.
- Driving the future with **cutting-edge, accessible AI** for everyone.
- Bridging the gap between research and real-world applications.
- Hosting a bustling, cheeky, and endlessly creative ecosystem that dares to **hug the future hard**.

---

Come for the **AI**, stay for the **community**, and maybe even make your computer a bit smarter (or at least better at memes).  

## Dive In:

**Website:** https://huggingface.co  
**Discord:** Where the magic and memes happen.  
**GitHub:** Peek under the hood of the AI revolution.  
**Join Us:** Because the future won't build itself.  

---

*Hugging Face — hugging your data, models, and sanity since forever.* 🤗

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [27]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [28]:
stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 12 relevant links


# Hugging Face: The AI Community Building the Future

The platform where the machine learning community collaborates on models, datasets, and applications. Welcome to the home of open collaboration, rapid iteration, and enough AI hype to power a small country—minus the bureaucracy.

## What Hugging Face is (in one snappy sentence)
A bustling hub for researchers, developers, and curious humans to create, discover, and collaborate on ML models, datasets, and apps—plus tools that actually make it easier to deploy AI in the real world.

## The toolkit you’ll find here
- Models: 2M+ models to browse, remix, or take for a spin
- Datasets: 500k+ datasets to train on, critique, and crowd-check
- Spaces: Live AI apps and demos you can run, customize, and learn from
- Inference: Endpoints and providers to deploy models at scale
- Storage Buckets: A place to stash your data and assets
- HuggingChat: The AI chat experience built for this community
- PRO & Enterprise: Team-ready plans and enterprise support
- Learn, Docs, Hardware: Resources to level up your skills and your infrastructure
- Community channels: Discord, Forum, GitHub, and a thriving ecosystem of collaborators

## Why customers love Hugging Face
- One-stop collaboration platform for models, datasets, and applications
- Access to a vast, evolving library of community-made AI solutions
- Enterprise-ready options with dedicated support and infrastructure
- Flexible deployment: public demos, hosted spaces, or private endpoints
- Transparent, community-driven development that helps you stay current in a fast-moving field

## For developers, researchers, and builders
- Rapid discovery: search 2M+ models and 500k+ datasets to find the right starting point
- Hands-on experimentation: run and customize AI apps in Spaces
- Reproducibility and remixability: collaborate openly with versioned models and datasets
- End-to-end workflows: from experimentation to deployment with robust tooling

## For investors and stakeholders
- A thriving, active ecosystem with constant updates and fresh ideas
- A clear path from research to real-world application through Spaces, endpoints, and enterprise offerings
- A platform that emphasizes collaboration, openness, and practical AI deployment

## Company culture and vibe
- Community-first: the platform is built by and for the ML community
- Open collaboration: models, datasets, and apps are shared, improved, and discussed
- Global and inclusive: a diverse ecosystem of contributors, researchers, and users
- Transparent progress: frequent updates, demos, and real-world use cases
- Practical focus: not just papers—tools and endpoints you can actually use

## Careers and opportunities
- Hugging Face emphasizes teamwork and a culture of collaboration
- Team & Enterprise and Hugging Face PRO indicate a strong emphasis on enterprise-friendly products
- For current openings and roles, check the company’s careers listings and team pages

## Brand vibe and assets
- Brand colors: HF Yellow and Orange tones to signal energy and optimism
  - #FFD21E
  - #FF9D00
- Official assets available for download (logos in multiple formats) for partner and contributor use

## Get involved
- Explore AI Apps and Spaces to see what people are building
- Browse 2M+ models and 500k+ datasets to fuel your projects
- Engage with the community on Discord, Forum, and GitHub
- If you’re an enterprise or team looking for scalable support and deployments, check out Hugging Face PRO and Enterprise options

Join Hugging Face and be part of the movement shaping how the world builds and uses AI—together, we’re not just imagining the future; we’re shipping it.

In [32]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 3 relevant links


# Hugging Face — The AI community building the future

The platform where the machine learning community collaborates on models, datasets, and applications. Basically a sci‑fi dream where people trade code, data, and vibes instead of stickers.

## What you can do here
- Browse 2M+ models. From Qwen to MiniMax, the shelf is crowded with fame, glory, and sometimes weird LoRAs.
- Explore Spaces. Run apps, generate music, make videos, and watch the AI do the heavy lifting while you pretend you’re the genius behind it.
- Commit to Datasets. Browse 500k+ datasets and decide whether your next breakthrough lives in a spreadsheet or in a rock-solid CSV.
- Manage Buckets. Storage for your AI collateral, because even models need a closet to call home.
- Tap into HuggingChat and other collaboration tools. A place where conversations with AI feel like hanging out with a very polite robot.
- Inference providers and endpoints. Scale your AI magic with enterprise-grade access, because big problems deserve big horsepower.
- Pro and Enterprise options. For teams who want more than just a playground—support, reliability, and all the bells and whistles.

Trending this week? You bet. The page showcases top models, spaces, and applications to spark ideas and fuel collaboration.

## The Home of Machine Learning
Create, discover and collaborate on ML better. The collaboration platform is built for teams, researchers, developers, and dreamers who want to build the future together.

- Models, Datasets, Spaces … plus all the bits that make ML life easier.
- The tagline says it best: “The AI community building the future.”
- Brand assets are available for download if you’re building something official (logos, colors, etc.).

## Why customers love it
- A massive library: Browse 2M+ models and 500k+ datasets. If your project is a long road, Hugging Face is a crowded highway.
- Quick experimentation: Run and share AI apps via Spaces, with real-time collaboration vibes.
- Enterprise-ready: Inference endpoints, providers, and dedicated enterprise support to keep production humming.
- Community-first ethos: It’s not just tools; it’s a community platform for collaboration on models, datasets, and applications.

## Culture and community
- Open, collaborative, and community-driven. The site highlights a platform built by and for the ML community.
- Multiple channels for engagement: Discord, Forum, GitHub, and more — because great ideas deserve multiple entry ramps.
- A spirit of accessibility: a home for researchers, developers, and teams to create and share widely.

## Careers and joining the team
- There’s a Team & Enterprise angle and a Hugging Face PRO path, signposting opportunities to collaborate with the growing community. If you’re hunting for roles, check the company’s pages for current openings and how you might contribute to the future of AI.

## Brand and visuals
- Official brand assets are available to download, including logo files and color palettes.
- Brand colors: HF Yellow (#FFD21E), HF Orange (#FF9D00), and a cool neutral (#6B7280) to keep things readable when your model’s ego gets too bright.
- The branding positions Hugging Face as the collaboration platform for the ML community.

## A quick reality check
- It’s the hub where “the AI community building the future” isn’t just a slogan — it’s the workflow. Models, datasets, spaces, and end-to-end collaboration all under one roof.
- If you’re a creator, researcher, investor, or someone who wants to recruit top ML talent, this is where the community meets the code, the data, and the edge cases.

## Ready to dive in?
Explore 2M+ models, browse 500k+ datasets, and check out Spaces that run the show. Whether you’re shipping a product, prototyping a research idea, or scouting talent, Hugging Face is your co-pilot in the wild west of AI.

Want the official assets? Brand pages have you covered. Want the community vibes? Discord, Forum, and Blog posts await. And yes, there’s a Pro and Enterprise path if you’re playing for keeps.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>